# Contrôles de la cible : ce qui affecte encore le défaut

**But** : comprendre ce qui fait encore varier le défaut de paiement (`dpnm`) dans la population laissée au machine learning, une fois retirés les clients en incident à M et en retard imminent. C'est une petite EDA sur la population restante, suivie de deux contrôles sur les populations retirées ou hors périmètre.

**Ce notebook est descriptif** : aucun modèle n'est entraîné, il ne choisit aucune variable (les variables des modèles sont choisies par l'élagage). Il mesure des taux de défaut, sur les **jeux d'entraînement seulement** :
- `data/ML/train_dataset.csv` pour la population du ML (le test du ML n'est jamais lu) ;
- `data/cleaned3_creditcard.csv` pour les clients hors périmètre (sans dette en septembre : ils ne sont dans aucun jeu de test) et pour la population en incident à M, prise dans le train du découpage de `05_02_EDA_incidents` (périmètre S12 d'abord : `BILL_AMT1 > 0`, plafond de 500 000 NT$ au plus ; puis 80/20, stratifié sur `dpnm`, `random_state=42`).

Les tranches de montants et de ratios sont fixées à l'avance (valeurs rondes), jamais calculées sur les clients ni cherchées sur la cible.

Convention temporelle : `BILL_AMTn` est la facture de fin du mois n ; `PAY_AMTn` est le paiement fait pendant le mois n, qui règle la facture du mois précédent `BILL_AMT(n+1)` ; `PAY_n` juge la facture `BILL_AMTn` sur le paiement du mois suivant `PAY_AMT(n-1)`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

train_ml = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")
cleaned3 = pd.read_csv(root_dir / "data" / "cleaned3_creditcard.csv")

# Train du découpage de 05_02_EDA_incidents : périmètre S12 d'abord, puis tirage 80/20 stratifié, graine 42
# (même ordre de lignes : cleaned3 garde les clients de cleaned2 dans le même ordre)
s12 = cleaned3[(cleaned3["BILL_AMT1"] > 0) & (cleaned3["LIMIT_BAL"] <= 500000)]
train_s12, _ = train_test_split(s12, test_size=0.2, stratify=s12["dpnm"], random_state=42)

TAUX_ML = train_ml["dpnm"].mean() * 100

def taux_par(d, colonne, ordre=None):
    # Taux de défaut par valeur d'une colonne : effectif, défauts, taux, écart au taux du train du ML (en points)
    t = d.groupby(colonne, observed=True)["dpnm"].agg(Clients="size", Défauts="sum")
    if ordre is not None:
        t = t.reindex([v for v in ordre if v in t.index])
    t["Part des clients (%)"] = (t["Clients"] / len(d) * 100).round(1)
    t["Taux de défaut (%)"] = (t["Défauts"] / t["Clients"] * 100).round(1)
    t["Écart au train du ML (pts)"] = (t["Taux de défaut (%)"] - TAUX_ML).round(1)
    return t

def montrer(t, titre):
    display(Markdown(f"**{titre}**"))
    display(t)

print(f"Train du ML : {len(train_ml)} clients, {int(train_ml['dpnm'].sum())} défauts, taux de défaut {TAUX_ML:.1f} %")
print(f"Train du découpage commun (S12) : {len(train_s12)} clients")

Train du ML : 17967 clients, 2481 défauts, taux de défaut 13.8 %
Train du découpage commun (S12) : 21761 clients


**Comment lire** : le taux de défaut du train du ML est la référence de tout le notebook. Dans chaque tableau qui suit, la colonne « Écart au train du ML » dit de combien de points un groupe est au-dessus (positif) ou au-dessous (négatif) de cette référence. Un écart sur un petit groupe (quelques dizaines de clients) n'est qu'un ordre de grandeur.

## 1. L'historique des retards

Les statuts à M gardés dans le ML, puis les indicateurs de risque passés : la famille **incident** (deux retards d'affilée ou un retard grave) et la famille **retard isolé** (un seul mois à 2, régularisé ensuite). Définitions : `docs/colonnes_creees.md`, section 2.

In [2]:
montrer(taux_par(train_ml, "STATUT_INCIDENT", ["Jamais en retard", "Retard régularisé", "Sorti de l'incident"]), "Statut à M")
montrer(taux_par(train_ml, "NB_MOIS_INCIDENT"), "Nombre de mois passés en incident (NB_MOIS_INCIDENT, tous les passages)")
montrer(taux_par(train_ml[train_ml["FLAG_INCIDENT"] == 1], "MOIS_SORTIE_INCIDENT"), "Clients sortis de l'incident : mois de la sortie (1 = septembre, 5 = mai)")
montrer(taux_par(train_ml, "NB_MOIS_RETARD"), "Nombre de mois de retard isolé (NB_MOIS_RETARD)")
montrer(taux_par(train_ml[train_ml["FLAG_RETARD"] == 1], "MOIS_SORTIE_RETARD"), "Clients avec un retard régularisé : mois de la régularisation (1 = septembre, 5 = mai)")
montrer(taux_par(train_ml, "CUMUL_INCIDENT"), "Nombre total de mois en retard (CUMUL_INCIDENT)")

**Statut à M**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
STATUT_INCIDENT,,,,,
Jamais en retard,14171,1573,78.9,11.1,-2.7
Retard régularisé,2024,413,11.3,20.4,6.6
Sorti de l'incident,1772,495,9.9,27.9,14.1


**Nombre de mois passés en incident (NB_MOIS_INCIDENT, tous les passages)**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
NB_MOIS_INCIDENT,,,,,
0,16195,1986,90.1,12.3,-1.5
1,829,157,4.6,18.9,5.1
2,506,159,2.8,31.4,17.6
3,250,95,1.4,38.0,24.2
4,164,72,0.9,43.9,30.1
5,23,12,0.1,52.2,38.4


**Clients sortis de l'incident : mois de la sortie (1 = septembre, 5 = mai)**

...

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
NB_MOIS_RETARD,,,,,
0,15714,2001,87.5,12.7,-1.1
1,2085,433,11.6,20.8,7.0
2,166,47,0.9,28.3,14.5
3,2,0,0.0,0.0,-13.8


**Clients avec un retard régularisé : mois de la régularisation (1 = septembre, 5 = mai)**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
MOIS_SORTIE_RETARD,,,,,
1.0,310,100,13.8,32.3,18.5
2.0,566,132,25.1,23.3,9.5
3.0,612,132,27.2,21.6,7.8
4.0,678,102,30.1,15.0,1.2
5.0,87,14,3.9,16.1,2.3


**Nombre total de mois en retard (CUMUL_INCIDENT)**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
CUMUL_INCIDENT,,,,,
0,14171,1573,78.9,11.1,-2.7
1,2521,484,14.0,19.2,5.4
2,774,220,4.3,28.4,14.6
3,311,117,1.7,37.6,23.8
4,167,75,0.9,44.9,31.1
5,23,12,0.1,52.2,38.4


**Comment lire** : une ligne par valeur, avec son effectif et son taux de défaut. Pour les mois de sortie, seuls les clients concernés sont comptés (un mois de sortie n'a de sens que si l'indicateur vaut 1) : 1 est la sortie la plus récente. On regarde si le taux monte avec le nombre de mois de retard, et s'il dépend de l'ancienneté de la sortie.

## 2. Les codifications de chaque mois

Taux de défaut selon la valeur de chaque codification, mois par mois (`PAY_1` = septembre, `PAY_6` = avril), puis selon la codification habituelle du client (`PAY_habituel`, retards regroupés en 2).

In [3]:
lignes = {}
for n in range(1, 7):
    t = taux_par(train_ml, f"PAY_{n}")
    lignes[f"PAY_{n}"] = t["Taux de défaut (%)"].astype(str) + " % (" + t["Clients"].astype(str) + ")"
montrer(pd.DataFrame(lignes).rename_axis("Codification").fillna("—"), "Taux de défaut par codification et par mois (effectif entre parenthèses)")
montrer(taux_par(train_ml, "PAY_habituel"), "Codification habituelle (PAY_habituel)")

**Taux de défaut par codification et par mois (effectif entre parenthèses)**

,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
Codification,,,,,,
-2,9.8 % (1532),9.5 % (1417),11.0 % (1848),12.3 % (2069),13.0 % (2231),13.6 % (2493)
-1,17.6 % (4774),14.9 % (4280),14.6 % (4094),14.3 % (3910),14.7 % (3802),15.1 % (3893)
0,12.8 % (11661),12.9 % (11747),12.2 % (10895),12.0 % (10675),12.2 % (10546),11.9 % (10150)
2,—,34.8 % (465),31.1 % (1094),29.3 % (1272),24.2 % (1346),23.9 % (1378)
3,—,49.0 % (51),42.9 % (28),32.3 % (31),53.1 % (32),40.0 % (40)
4,—,60.0 % (5),28.6 % (7),75.0 % (8),50.0 % (6),60.0 % (5)
5,—,50.0 % (2),0.0 % (1),0.0 % (1),66.7 % (3),0.0 % (3)
6,—,—,—,0.0 % (1),—,60.0 % (5)
7,—,—,—,—,100.0 % (1),—


**Codification habituelle (PAY_habituel)**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
PAY_habituel,,,,,
-2,1735,199,9.7,11.5,-2.3
-1,4335,694,24.1,16.0,2.2
0,11622,1464,64.7,12.6,-1.2
2,275,124,1.5,45.1,31.3


**Comment lire** : une colonne par mois, une ligne par valeur de la codification ; chaque case donne le taux de défaut et, entre parenthèses, le nombre de clients. « — » : valeur absente ce mois-là. Les valeurs de 3 et plus ne concernent que quelques clients : leurs taux sont des ordres de grandeur.

## 3. Le comportement de paiement et l'usage du plafond

Le type d'usage de la carte (construit sur les seuls paiements), le taux de remboursement sur la période, l'utilisation du plafond en septembre, la vie du compte.

In [4]:
TYPES_USAGE = ["Ne paie rien", "Client en difficulté", "Crédit lent", "Crédit rapide", "Usage mixte", "Paiement comptant", "Première facture"]
montrer(taux_par(train_ml, "TYPE_USAGE", TYPES_USAGE), "Type d'usage de la carte (TYPE_USAGE)")

tranches_rembourse = pd.cut(train_ml["ratio_PAY_BILL_global"], bins=[-0.1, 0, 5, 10, 25, 50, 95, 200],
                            labels=["0 %", "0-5 %", "5-10 %", "10-25 %", "25-50 %", "50-95 %", "plus de 95 %"])
montrer(taux_par(train_ml.assign(Tranche=tranches_rembourse), "Tranche"), "Part du dû remboursée sur la période (ratio_PAY_BILL_global), vides exclus")

tranches_plafond = pd.cut(train_ml["ratio_BILL_LIMIT1"], bins=[-0.1, 10, 25, 50, 75, 90, 100, 200],
                          labels=["0-10 %", "10-25 %", "25-50 %", "50-75 %", "75-90 %", "90-100 %", "au-delà du plafond"])
montrer(taux_par(train_ml.assign(Tranche=tranches_plafond), "Tranche"), "Utilisation du plafond en septembre (ratio_BILL_LIMIT1)")

vie = np.select([train_ml["FLAG_OUVERTURE"] == 1, train_ml["FLAG_DEGEL"] == 1], ["Ouverture pendant la période", "Réveil après un gel"], "Compte actif")
montrer(taux_par(train_ml.assign(Vie=vie), "Vie", ["Compte actif", "Ouverture pendant la période", "Réveil après un gel"]), "Vie du compte")

**Type d'usage de la carte (TYPE_USAGE)**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
TYPE_USAGE,,,,,
Ne paie rien,182,69,1.0,37.9,24.1
Client en difficulté,232,34,1.3,14.7,0.9
Crédit lent,6770,894,37.7,13.2,-0.6
Crédit rapide,2956,415,16.5,14.0,0.2
Usage mixte,1994,265,11.1,13.3,-0.5
Paiement comptant,5503,748,30.6,13.6,-0.2
Première facture,330,56,1.8,17.0,3.2


**Part du dû remboursée sur la période (ratio_PAY_BILL_global), vides exclus**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
Tranche,,,,,
0 %,29,12,0.2,41.4,27.6
0-5 %,51,7,0.3,13.7,-0.1
5-10 %,288,43,1.6,14.9,1.1
10-25 %,6380,819,35.5,12.8,-1.0
25-50 %,2952,448,16.4,15.2,1.4
50-95 %,2241,330,12.5,14.7,0.9
plus de 95 %,5709,767,31.8,13.4,-0.4


**Utilisation du plafond en septembre (ratio_BILL_LIMIT1)**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
Tranche,,,,,
0-10 %,6524,962,36.3,14.7,0.9
10-25 %,1925,185,10.7,9.6,-4.2
25-50 %,2254,244,12.5,10.8,-3.0
50-75 %,2144,271,11.9,12.6,-1.2
75-90 %,1573,232,8.8,14.7,0.9
90-100 %,2431,379,13.5,15.6,1.8
au-delà du plafond,1116,208,6.2,18.6,4.8


**Vie du compte**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
Vie,,,,,
Compte actif,16133,2165,89.8,13.4,-0.4
Ouverture pendant la période,1522,269,8.5,17.7,3.9
Réveil après un gel,312,47,1.7,15.1,1.3


**Comment lire** : une ligne par groupe, rangés dans un ordre fixe (du client qui ne paie rien au client qui paie comptant ; de la plus faible à la plus forte utilisation du plafond). Les tranches sont des valeurs rondes fixées à l'avance. Le taux de remboursement est vide pour les clients qui n'avaient rien à rembourser sur la période : ils ne sont pas comptés dans ce tableau.

## 4. Le plafond et la démographie

In [5]:
tranches_limite = pd.cut(train_ml["LIMIT_BAL"], bins=[0, 50000, 100000, 200000, 300000, 500000],
                         labels=["jusqu'à 50 000", "50 000 à 100 000", "100 000 à 200 000", "200 000 à 300 000", "300 000 à 500 000"])
montrer(taux_par(train_ml.assign(Tranche=tranches_limite), "Tranche"), "Plafond (LIMIT_BAL, NT$)")

tranches_age = pd.cut(train_ml["AGE"], bins=[20, 25, 30, 35, 40, 50, 80], labels=["21-25", "26-30", "31-35", "36-40", "41-50", "51+"])
montrer(taux_par(train_ml.assign(Tranche=tranches_age), "Tranche"), "Âge (mêmes tranches que AGE_BUCKET)")
montrer(taux_par(train_ml, "SEX").rename(index={1: "1 (homme)", 2: "2 (femme)"}), "Sexe")
montrer(taux_par(train_ml, "EDUCATION").rename(index={1: "1 (troisième cycle)", 2: "2 (université)", 3: "3 (lycée)", 4: "4 (autres)"}), "Études")
montrer(taux_par(train_ml, "MARRIAGE").rename(index={1: "1 (marié)", 2: "2 (célibataire)", 3: "3 (autres)"}), "État civil")

**Plafond (LIMIT_BAL, NT$)**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
Tranche,,,,,
jusqu'à 50 000,4178,832,23.3,19.9,6.1
50 000 à 100 000,2803,436,15.6,15.6,1.8
100 000 à 200 000,4866,638,27.1,13.1,-0.7
200 000 à 300 000,3237,347,18.0,10.7,-3.1
300 000 à 500 000,2883,228,16.0,7.9,-5.9


**Âge (mêmes tranches que AGE_BUCKET)**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
Tranche,,,,,
21-25,2254,381,12.5,16.9,3.1
26-30,4344,549,24.2,12.6,-1.2
31-35,3514,437,19.6,12.4,-1.4
36-40,2961,404,16.5,13.6,-0.2
41-50,3623,520,20.2,14.4,0.6
51+,1271,190,7.1,14.9,1.1


**Sexe**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
SEX,,,,,
1 (homme),7030,1085,39.1,15.4,1.6
2 (femme),10937,1396,60.9,12.8,-1.0


**Études**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
EDUCATION,,,,,
1 (troisième cycle),6405,794,35.6,12.4,-1.4
2 (université),8429,1245,46.9,14.8,1.0
3 (lycée),2818,427,15.7,15.2,1.4
4 (autres),315,15,1.8,4.8,-9.0


**État civil**

,Clients,Défauts,Part des clients (%),Taux de défaut (%),Écart au train du ML (pts)
MARRIAGE,,,,,
1 (marié),8122,1182,45.2,14.6,0.8
2 (célibataire),9624,1261,53.6,13.1,-0.7
3 (autres),221,38,1.2,17.2,3.4


**Comment lire** : même lecture que plus haut. Les libellés des valeurs de `SEX`, `EDUCATION` et `MARRIAGE` sont ceux de la documentation du dataset (après le nettoyage, les valeurs non documentées sont regroupées dans « autres »). Le plafond est fixé par la banque à l'ouverture du compte : il reflète en partie son propre jugement du client.

## 5. Hors périmètre : ne rien devoir protège-t-il du défaut ?

Clients de `cleaned3` sans dette en septembre (`BILL_AMT1 <= 0`), hors du périmètre du ML : ils ne sont dans aucun jeu de test.

In [6]:
factures = [f"BILL_AMT{n}" for n in range(1, 7)]
lignes = []
for libelle, d in [("Tous les clients de cleaned3", cleaned3),
                   ("Sans dette en septembre (BILL_AMT1 ≤ 0)", cleaned3[cleaned3["BILL_AMT1"] <= 0]),
                   ("Sans aucune dette sur les six mois (toutes les factures ≤ 0)", cleaned3[(cleaned3[factures] <= 0).all(axis=1)])]:
    lignes.append({"Population": libelle, "Clients": len(d), "Défauts": int(d["dpnm"].sum()), "Taux de défaut (%)": round(d["dpnm"].mean() * 100, 1)})
montrer(pd.DataFrame(lignes).set_index("Population"), "Taux de défaut selon la dette")

**Taux de défaut selon la dette**

,Clients,Défauts,Taux de défaut (%)
Population,,,
Tous les clients de cleaned3,28851,6272,21.7
Sans dette en septembre (BILL_AMT1 ≤ 0),1649,300,18.2
Sans aucune dette sur les six mois (toutes les factures ≤ 0),25,6,24.0


**Comment lire** : la première ligne (tous les clients, test compris) ne sert que de repère. Si les clients sans dette font défaut presque aussi souvent que la moyenne, ne rien devoir ne protège pas du défaut tel qu'il est codé : un indice que la cible n'est pas (seulement) un défaut de paiement observé (limite connue : `docs/contexte.md`).

## 6. Population en incident à M : payer change-t-il le défaut ?

Clients en incident à M du **train du découpage commun** (le test de ce découpage a déjà servi, une fois, à valider la définition de la population en incident). Rappel : le paiement fait en septembre (`PAY_AMT1`) règle la facture d'août (`BILL_AMT2`) ; celui d'août (`PAY_AMT2`), la facture de juillet (`BILL_AMT3`).

In [7]:
incident = train_s12[train_s12["STATUT_INCIDENT"] == "En incident"]
groupes = [
    ("En incident à M, tous", incident),
    ("A payé en août et en septembre (PAY_AMT2 > 0 et PAY_AMT1 > 0)", incident[(incident["PAY_AMT1"] > 0) & (incident["PAY_AMT2"] > 0)]),
    ("N'a payé ni en août ni en septembre", incident[(incident["PAY_AMT1"] == 0) & (incident["PAY_AMT2"] == 0)]),
    ("A réglé en septembre au moins la facture d'août (PAY_AMT1 ≥ BILL_AMT2 > 0)", incident[(incident["BILL_AMT2"] > 0) & (incident["PAY_AMT1"] >= incident["BILL_AMT2"])]),
    ("Incident court : 1 ou 2 mois (NB_MOIS_INCIDENT ≤ 2)", incident[incident["NB_MOIS_INCIDENT"] <= 2]),
    ("Six mois en incident (NB_MOIS_INCIDENT = 6)", incident[incident["NB_MOIS_INCIDENT"] == 6]),
]
lignes = [{"Groupe": g, "Clients": len(d), "Défauts": int(d["dpnm"].sum()),
           "Taux de défaut (%)": round(d["dpnm"].mean() * 100, 1) if len(d) else np.nan} for g, d in groupes]
montrer(pd.DataFrame(lignes).set_index("Groupe"), "Population en incident à M (train du découpage commun) : taux de défaut par groupe")

**Population en incident à M (train du découpage commun) : taux de défaut par groupe**

,Clients,Défauts,Taux de défaut (%)
Groupe,,,
"En incident à M, tous",3019,1779,58.9
A payé en août et en septembre (PAY_AMT2 > 0 et PAY_AMT1 > 0),1007,612,60.8
N'a payé ni en août ni en septembre,350,238,68.0
A réglé en septembre au moins la facture d'août (PAY_AMT1 ≥ BILL_AMT2 > 0),3,0,0.0
Incident court : 1 ou 2 mois (NB_MOIS_INCIDENT ≤ 2),698,290,41.5
Six mois en incident (NB_MOIS_INCIDENT = 6),1095,772,70.5


**Comment lire** : la première ligne donne le taux de défaut de toute la population en incident à M du train ; les suivantes, celui de quelques groupes définis à l'avance par les paiements ou par la durée de l'incident. Un groupe nettement sous la première ligne serait une piste pour sortir des clients de l'incident (piste à part, après le ML principal) ; un écart sur un petit groupe n'est qu'un ordre de grandeur.